In [0]:
from pyspark.sql.functions import col, sum as _sum, when, lit, countDistinct, current_timestamp
import pandas as pd
import getpass

# ====================================================================
# 1. SNOWFLAKE CONNECTION
# ====================================================================
sf_password = getpass.getpass("Enter Snowflake Password: ")
sfOptions = {
    "host": "iovnsqz-sf95102.snowflakecomputing.com",
    "sfUser": "madaladevendra",
    "sfPassword": sf_password,
    "sfDatabase": "HEALTHCARE_RCM",
    "sfSchema": "GOLD",
    "sfWarehouse": "RCM_WH"
}

LAYER_NAME = "Bronze"

# ====================================================================
# 2. BRONZE CRITICAL RULES
# ====================================================================
bronze_rules = {
    "Beneficiary": {
        "DESYNPUF_ID": col("DESYNPUF_ID").isNull() | col("DESYNPUF_ID").like("% %"),
        "BENE_SEX_IDENT_CD": col("BENE_SEX_IDENT_CD") == "9",
        "BENE_BIRTH_DT": col("BENE_BIRTH_DT") > 20101231,
        "BENE_DEATH_DT": col("BENE_DEATH_DT") < col("BENE_BIRTH_DT"),
        "SP_STATE_CODE": col("SP_STATE_CODE") == "999"
    },
    "Inpatient": {
        "CLM_ID": col("CLM_ID").isNull(),
        "CLM_ADMSN_DT": col("CLM_ADMSN_DT") > 20101231,
        "CLM_PMT_AMT": col("CLM_PMT_AMT") < 0,
        "PRVDR_NUM": col("PRVDR_NUM").isNull() | col("PRVDR_NUM").like("% %"),
        "ICD9_DGNS_CD_1": (col("ICD9_DGNS_CD_1") == "INVALID_ICD") | (col("CLM_DRG_CD") == "INV")
    },
    "Outpatient": {
        "CLM_ID": col("CLM_ID").isNull(),
        "CLM_FROM_DT": col("CLM_FROM_DT") > 20101231,
        "CLM_PMT_AMT": col("CLM_PMT_AMT") < 0,
        "PRVDR_NUM": col("PRVDR_NUM").isNull() | col("PRVDR_NUM").like("% %"),
        "HCPCS_CD_1": col("HCPCS_CD_1") == "INV_HCPCS"
    },
    "Carrier": {
        "CLM_ID": col("CLM_ID").isNull(),
        "CLM_FROM_DT": col("CLM_FROM_DT") > 20101231,
        "PRF_PHYSN_NPI_1": col("PRF_PHYSN_NPI_1").isNull(),
        "ICD9_DGNS_CD_1": (col("ICD9_DGNS_CD_1") == "INVALID_ICD") | (col("HCPCS_CD_1") == "INV_HCPCS"),
        "LINE_NCH_PMT_AMT_1": col("LINE_NCH_PMT_AMT_1") < 0
    },
    "Pharmacy (PDE)": {
        "PDE_ID": col("PDE_ID").isNull(),
        "SRVC_DT": col("SRVC_DT") > 20101231,
        "QTY_DSPNSD_NUM": (col("QTY_DSPNSD_NUM") < 0) | (col("DAYS_SUPLY_NUM") < 0),
        "TOT_RX_CST_AMT": (col("PTNT_PAY_AMT") < 0) | (col("TOT_RX_CST_AMT") < 0),
        "PROD_SRVC_ID": col("PROD_SRVC_ID").isNull() | col("PROD_SRVC_ID").like("% %") | (col("PROD_SRVC_ID") == "INV_NDC")
    }
}

# ====================================================================
# 3. DYNAMIC CORE ENGINE
# ====================================================================
def scan_all_columns_dq(df, dataset_name, rules_dict):
    total_rows = df.count()
    if total_rows == 0: return []
    
    all_columns = df.columns
    agg_exprs = []
    
    for c_name in all_columns:
        agg_exprs.append(_sum(when(col(c_name).isNull(), 1).otherwise(0)).alias(f"{c_name}_nulls"))
        agg_exprs.append(countDistinct(col(c_name)).alias(f"{c_name}_uniques"))
        
        if c_name in rules_dict:
            agg_exprs.append(_sum(when(rules_dict[c_name], 1).otherwise(0)).alias(f"{c_name}_anomalies"))
        else:
            agg_exprs.append(lit(0).alias(f"{c_name}_anomalies"))
            
    metrics = df.select(*agg_exprs).collect()[0].asDict()
    
    flat_data = []
    for c_name in all_columns:
        nulls = metrics[f"{c_name}_nulls"]
        uniques = metrics[f"{c_name}_uniques"]
        anomalies = metrics[f"{c_name}_anomalies"]
        valid_count = total_rows - anomalies
        
        comp = ((total_rows - nulls) / total_rows) * 100 if total_rows > 0 else 0
        uniq = (uniques / total_rows) * 100 if total_rows > 0 else 0
        yld = (valid_count / total_rows) * 100 if total_rows > 0 else 0
        acc = ((total_rows - anomalies) / total_rows) * 100 if total_rows > 0 else 0
        
        flat_data.append({
            "LAYER": LAYER_NAME,
            "DATASET_NAME": dataset_name,
            "COLUMN_NAME": c_name,
            "IS_CRITICAL": "Yes" if c_name in rules_dict else "No",
            "TOTAL_ROWS": total_rows,
            "NULL_COUNT": nulls,
            "UNIQUE_COUNT": uniques,
            "VALID_COUNT": valid_count,
            "ANOMALY_COUNT": anomalies,
            "COMPLETENESS_SCORE": round(comp, 2),
            "UNIQUENESS_SCORE": round(uniq, 2),
            "YIELD_SCORE": round(yld, 2),
            "ACCURACY_SCORE": round(acc, 2),
            "COMPOSITE_SCORE": round((comp + uniq + yld + acc) / 4, 2)
        })
    return flat_data

# ====================================================================
# 4. EXECUTE & OVERWRITE TO SNOWFLAKE
# ====================================================================
print(f"Executing {LAYER_NAME} Scans...")
all_metrics = []
all_metrics.extend(scan_all_columns_dq(spark.table("healthcare_rcm.bronze.raw_beneficiary"), "Beneficiary", bronze_rules["Beneficiary"]))
all_metrics.extend(scan_all_columns_dq(spark.table("healthcare_rcm.bronze.raw_inpatient_claims"), "Inpatient", bronze_rules["Inpatient"]))
all_metrics.extend(scan_all_columns_dq(spark.table("healthcare_rcm.bronze.raw_outpatient_claims"), "Outpatient", bronze_rules["Outpatient"]))
all_metrics.extend(scan_all_columns_dq(spark.table("healthcare_rcm.bronze.raw_carrier_claims"), "Carrier", bronze_rules["Carrier"]))
all_metrics.extend(scan_all_columns_dq(spark.table("healthcare_rcm.bronze.raw_prescription_drug"), "Pharmacy (PDE)", bronze_rules["Pharmacy (PDE)"]))

dq_df = spark.createDataFrame(pd.DataFrame(all_metrics)).withColumn("AUDIT_TIMESTAMP", current_timestamp())

# OVERWRITE for Bronze to reset/create the table cleanly
dq_df.write.format("snowflake").options(**sfOptions).option("dbtable", "ENTERPRISE_DQ_AUDIT_LOG").mode("overwrite").save()
print("✅ Bronze data successfully written to Snowflake!")

# ====================================================================
# 5. RENDER NATIVE DATABRICKS UI WITH CUSTOM THRESHOLDS
# ====================================================================
def get_color(score):
    if score >= 85.0: 
        return '#0D5C2F'   # Dark Green (85% - 100%)
    elif score >= 77.77: 
        return '#1e8e3e'   # Standard Green (77.77% - 85%)
    else: 
        return '#d93025'   # Red (0 - 77.7%)

bronze_layer_score = sum([m["COMPOSITE_SCORE"] for m in all_metrics]) / len(all_metrics) if all_metrics else 0.0

tables_html = ""
datasets = ["Beneficiary", "Inpatient", "Outpatient", "Carrier", "Pharmacy (PDE)"]

for ds in datasets:
    ds_metrics = [m for m in all_metrics if m["DATASET_NAME"] == ds]
    if not ds_metrics: continue
    
    ds_avg_score = sum([m["COMPOSITE_SCORE"] for m in ds_metrics]) / len(ds_metrics)
    
    tables_html += f"""
    <div style="background: white; padding: 15px; border-radius: 8px; border-top: 4px solid #b71c1c; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin-bottom: 25px;">
        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #eee; padding-bottom: 10px; margin-bottom: 10px;">
            <h3 style="margin: 0; color: #b71c1c;">{ds} Dataset</h3>
            <h2 style="margin: 0; color: {get_color(ds_avg_score)};">{round(ds_avg_score, 2)}% Avg</h2>
        </div>
        <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 13px;">
            <tr style="background-color: #f8d7da;">
                <th style="padding: 8px; border-bottom: 1px solid #ddd; color: #721c24;">Column Name</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Nulls</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Unique</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Valid</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Anomalies</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Composite Score</th>
            </tr>
    """
    for c in ds_metrics:
        anomaly_text = f"<span style='color: #1e8e3e;'>✅ 0</span>" if c['ANOMALY_COUNT'] == 0 else f"<span style='color: #d93025;'>{c['ANOMALY_COUNT']:,}</span>"
        null_text = f"<span style='color: #1e8e3e;'>✅ 0</span>" if c['NULL_COUNT'] == 0 else f"<span style='color: #5f6368;'>{c['NULL_COUNT']:,}</span>"

        tables_html += f"""
            <tr>
                <td style="padding: 8px; border-bottom: 1px solid #eee; font-family: monospace;"><strong>{c['COLUMN_NAME']}</strong></td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right;">{null_text}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right; color: #1a73e8;">{c['UNIQUE_COUNT']:,}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right; color: #1e8e3e;">{c['VALID_COUNT']:,}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right;">{anomaly_text}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right; font-weight: bold; color: {get_color(c['COMPOSITE_SCORE'])}; font-size: 15px;">{c['COMPOSITE_SCORE']}%</td>
            </tr>
        """
    tables_html += "</table></div>"

html_code = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; padding: 20px; background-color: #f0f2f5; border-radius: 8px;">
    <div style="text-align: center; margin-bottom: 30px;">
        <h1 style="color: #b71c1c; margin-bottom: 5px;">Data Quality Control Tower</h1>
        <p style="color: #5f6368; margin-top: 0; font-size: 16px;">Medallion Architecture: <b>Bronze Layer</b> | Raw Ingestion Scan</p>
    </div>
    <div style="background: white; padding: 25px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); margin-bottom: 30px; text-align: center; border: 1px solid #e0e0e0; border-top: 6px solid #b71c1c;">
        <h3 style="margin: 0; color: #5f6368; text-transform: uppercase; letter-spacing: 1px;">Overall Bronze Composite Score</h3>
        <h1 style="margin: 15px 0 0 0; color: {get_color(bronze_layer_score)}; font-size: 54px;">
            {round(bronze_layer_score, 2)}%
        </h1>
        <p style="color: #5f6368; font-size: 14px; margin-top: 10px; font-weight: bold;">📥 Raw Ingestion Validation Complete.</p>
    </div>
    {tables_html}
</div>
"""

displayHTML(html_code)

Enter Snowflake Password:  [REDACTED]

Executing Bronze Scans...
✅ Bronze data successfully written to Snowflake!


Column Name,Nulls,Unique,Valid,Anomalies,Composite Score
DESYNPUF_ID,15,"116,381","343,542",315,83.41%
BENE_BIRTH_DT,✅ 0,901,"343,707",150,75.04%
BENE_DEATH_DT,"338,424",37,"343,705",152,50.38%
BENE_SEX_IDENT_CD,✅ 0,3,"343,557",300,74.96%
BENE_RACE_CD,✅ 0,4,"343,857",✅ 0,75.0%
BENE_ESRD_IND,✅ 0,2,"343,857",✅ 0,75.0%
SP_STATE_CODE,✅ 0,53,"343,557",300,74.96%
BENE_COUNTY_CD,✅ 0,310,"343,857",✅ 0,75.02%
BENE_HI_CVRAGE_TOT_MONS,✅ 0,13,"343,857",✅ 0,75.0%
BENE_SMI_CVRAGE_TOT_MONS,✅ 0,13,"343,857",✅ 0,75.0%


In [0]:
from pyspark.sql.functions import col, sum as _sum, when, lit, countDistinct, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
import pandas as pd
import getpass

# ====================================================================
# 1. SNOWFLAKE CONNECTION
# ====================================================================
sf_password = getpass.getpass("Enter Snowflake Password: ")
sfOptions = {
    "host": "iovnsqz-sf95102.snowflakecomputing.com",
    "sfUser": "madaladevendra",
    "sfPassword": sf_password,
    "sfDatabase": "HEALTHCARE_RCM",
    "sfSchema": "GOLD",
    "sfWarehouse": "RCM_WH"
}

LAYER_NAME = "Silver"

# ====================================================================
# 2. SILVER CRITICAL RULES
# ====================================================================
silver_rules = {
    "Beneficiary": {
        "DESYNPUF_ID": col("DESYNPUF_ID").isNull() | col("DESYNPUF_ID").like("% %"),
        "BENE_SEX_IDENT_CD": col("BENE_SEX_IDENT_CD") == "9",
        "BENE_BIRTH_DT": col("BENE_BIRTH_DT") > lit("2010-12-31").cast("date"),
        "BENE_DEATH_DT": col("BENE_DEATH_DT") < col("BENE_BIRTH_DT"),
        "SP_STATE_CODE": col("SP_STATE_CODE") == "999"
    },
    "Inpatient": {
        "CLM_ID": col("CLM_ID").isNull(),
        "CLM_ADMSN_DT": col("CLM_ADMSN_DT") > lit("2010-12-31").cast("date"),
        "CLM_PMT_AMT": col("CLM_PMT_AMT") < 0,
        "PRVDR_NUM": col("PRVDR_NUM").isNull() | col("PRVDR_NUM").like("% %"),
        "ICD9_DGNS_CD_1": (col("ICD9_DGNS_CD_1") == "INVALID_ICD") | (col("CLM_DRG_CD") == "INV")
    },
    "Outpatient": {
        "CLM_ID": col("CLM_ID").isNull(),
        "CLM_FROM_DT": col("CLM_FROM_DT") > lit("2010-12-31").cast("date"),
        "CLM_PMT_AMT": col("CLM_PMT_AMT") < 0,
        "PRVDR_NUM": col("PRVDR_NUM").isNull() | col("PRVDR_NUM").like("% %"),
        "HCPCS_CD_1": col("HCPCS_CD_1") == "INV_HCPCS"
    },
    "Carrier": {
        "CLM_ID": col("CLM_ID").isNull(),
        "CLM_FROM_DT": col("CLM_FROM_DT") > 20101231,
        "PRF_PHYSN_NPI_1": col("PRF_PHYSN_NPI_1").isNull(),
        "ICD9_DGNS_CD_1": (col("ICD9_DGNS_CD_1") == "INVALID_ICD") | (col("HCPCS_CD_1") == "INV_HCPCS"),
        "LINE_NCH_PMT_AMT_1": col("LINE_NCH_PMT_AMT_1") < 0
    },
    "Pharmacy (PDE)": {
        "PDE_ID": col("PDE_ID").isNull(),
        "SRVC_DT": col("SRVC_DT") > lit("2010-12-31").cast("date"),
        "QTY_DSPNSD_NUM": (col("QTY_DSPNSD_NUM") < 0) | (col("DAYS_SUPLY_NUM") < 0),
        "TOT_RX_CST_AMT": (col("PTNT_PAY_AMT") < 0) | (col("TOT_RX_CST_AMT") < 0),
        "PROD_SRVC_ID": col("PROD_SRVC_ID").isNull() | col("PROD_SRVC_ID").like("% %") | (col("PROD_SRVC_ID") == "INV_NDC")
    }
}

# ====================================================================
# 3. DYNAMIC CORE ENGINE
# ====================================================================
def scan_all_columns_dq(df, dataset_name, rules_dict):
    total_rows = df.count()
    if total_rows == 0: return []
    
    all_columns = df.columns
    agg_exprs = []
    
    for c_name in all_columns:
        agg_exprs.append(_sum(when(col(c_name).isNull(), 1).otherwise(0)).alias(f"{c_name}_nulls"))
        agg_exprs.append(countDistinct(col(c_name)).alias(f"{c_name}_uniques"))
        
        if c_name in rules_dict:
            agg_exprs.append(_sum(when(rules_dict[c_name], 1).otherwise(0)).alias(f"{c_name}_anomalies"))
        else:
            agg_exprs.append(lit(0).alias(f"{c_name}_anomalies"))
            
    metrics = df.select(*agg_exprs).collect()[0].asDict()
    
    flat_data = []
    for c_name in all_columns:
        nulls = metrics[f"{c_name}_nulls"]
        uniques = metrics[f"{c_name}_uniques"]
        anomalies = metrics[f"{c_name}_anomalies"]
        valid_count = total_rows - anomalies
        
        comp = ((total_rows - nulls) / total_rows) * 100 if total_rows > 0 else 0
        uniq = (uniques / total_rows) * 100 if total_rows > 0 else 0
        yld = (valid_count / total_rows) * 100 if total_rows > 0 else 0
        acc = ((total_rows - anomalies) / total_rows) * 100 if total_rows > 0 else 0
        
        flat_data.append({
            "LAYER": LAYER_NAME,
            "DATASET_NAME": dataset_name,
            "COLUMN_NAME": c_name,
            "IS_CRITICAL": "Yes" if c_name in rules_dict else "No",
            "TOTAL_ROWS": total_rows,
            "NULL_COUNT": nulls,
            "UNIQUE_COUNT": uniques,
            "VALID_COUNT": valid_count,
            "ANOMALY_COUNT": anomalies,
            "COMPLETENESS_SCORE": round(comp, 2),
            "UNIQUENESS_SCORE": round(uniq, 2),
            "YIELD_SCORE": round(yld, 2),
            "ACCURACY_SCORE": round(acc, 2),
            "COMPOSITE_SCORE": round((comp + uniq + yld + acc) / 4, 2)
        })
    return flat_data

# ====================================================================
# 4. STRICT SCHEMA TO PREVENT PANDAS TYPE MISMATCH ON APPEND
# ====================================================================
dq_schema = StructType([
    StructField("LAYER", StringType(), True),
    StructField("DATASET_NAME", StringType(), True),
    StructField("COLUMN_NAME", StringType(), True),
    StructField("IS_CRITICAL", StringType(), True),
    StructField("TOTAL_ROWS", LongType(), True),
    StructField("NULL_COUNT", LongType(), True),
    StructField("UNIQUE_COUNT", LongType(), True),
    StructField("VALID_COUNT", LongType(), True),
    StructField("ANOMALY_COUNT", LongType(), True),
    StructField("COMPLETENESS_SCORE", DoubleType(), True),
    StructField("UNIQUENESS_SCORE", DoubleType(), True),
    StructField("YIELD_SCORE", DoubleType(), True),
    StructField("ACCURACY_SCORE", DoubleType(), True),
    StructField("COMPOSITE_SCORE", DoubleType(), True)
])

# ====================================================================
# 5. EXECUTE & APPEND TO SNOWFLAKE
# ====================================================================
print(f"Executing {LAYER_NAME} Scans...")
all_metrics = []
all_metrics.extend(scan_all_columns_dq(spark.table("healthcare_rcm.silver.silver_beneficiary"), "Beneficiary", silver_rules["Beneficiary"]))
all_metrics.extend(scan_all_columns_dq(spark.table("healthcare_rcm.silver.silver_inpatient"), "Inpatient", silver_rules["Inpatient"]))
all_metrics.extend(scan_all_columns_dq(spark.table("healthcare_rcm.silver.silver_outpatient"), "Outpatient", silver_rules["Outpatient"]))
all_metrics.extend(scan_all_columns_dq(spark.table("healthcare_rcm.silver.silver_carrier"), "Carrier", silver_rules["Carrier"]))
all_metrics.extend(scan_all_columns_dq(spark.table("healthcare_rcm.silver.silver_pde"), "Pharmacy (PDE)", silver_rules["Pharmacy (PDE)"]))

# Create Spark DF using the STRICT SCHEMA
dq_df = spark.createDataFrame(pd.DataFrame(all_metrics), schema=dq_schema).withColumn("AUDIT_TIMESTAMP", current_timestamp())

# APPEND for Silver
dq_df.write.format("snowflake").options(**sfOptions).option("dbtable", "ENTERPRISE_DQ_AUDIT_LOG").mode("append").save()
print("✅ Silver data successfully appended to Snowflake!")

# ====================================================================
# 6. RENDER NATIVE DATABRICKS UI WITH CUSTOM THRESHOLDS
# ====================================================================
def get_color(score):
    if score >= 85.0: 
        return '#0D5C2F'   # Dark Green (85% - 100%)
    elif score >= 77.77: 
        return '#1e8e3e'   # Standard Green (77.77% - 85%)
    else: 
        return '#d93025'   # Red (0 - 77.7%)

silver_layer_score = sum([m["COMPOSITE_SCORE"] for m in all_metrics]) / len(all_metrics) if all_metrics else 0.0

tables_html = ""
datasets = ["Beneficiary", "Inpatient", "Outpatient", "Carrier", "Pharmacy (PDE)"]

for ds in datasets:
    ds_metrics = [m for m in all_metrics if m["DATASET_NAME"] == ds]
    if not ds_metrics: continue
    
    ds_avg_score = sum([m["COMPOSITE_SCORE"] for m in ds_metrics]) / len(ds_metrics)
    
    tables_html += f"""
    <div style="background: white; padding: 15px; border-radius: 8px; border-top: 4px solid #1a73e8; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin-bottom: 25px;">
        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #eee; padding-bottom: 10px; margin-bottom: 10px;">
            <h3 style="margin: 0; color: #1a237e;">{ds} Dataset</h3>
            <h2 style="margin: 0; color: {get_color(ds_avg_score)};">{round(ds_avg_score, 2)}% Avg</h2>
        </div>
        <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 13px;">
            <tr style="background-color: #e8f0fe;">
                <th style="padding: 8px; border-bottom: 1px solid #ddd; color: #1a73e8;">Column Name</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Nulls</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Unique</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Valid</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Anomalies</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Composite Score</th>
            </tr>
    """
    for c in ds_metrics:
        anomaly_text = f"<span style='color: #1e8e3e;'>✅ 0</span>" if c['ANOMALY_COUNT'] == 0 else f"<span style='color: #d93025;'>{c['ANOMALY_COUNT']:,}</span>"
        null_text = f"<span style='color: #1e8e3e;'>✅ 0</span>" if c['NULL_COUNT'] == 0 else f"<span style='color: #5f6368;'>{c['NULL_COUNT']:,}</span>"

        tables_html += f"""
            <tr>
                <td style="padding: 8px; border-bottom: 1px solid #eee; font-family: monospace;"><strong>{c['COLUMN_NAME']}</strong></td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right;">{null_text}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right; color: #1a73e8;">{c['UNIQUE_COUNT']:,}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right; color: #1e8e3e;">{c['VALID_COUNT']:,}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right;">{anomaly_text}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right; font-weight: bold; color: {get_color(c['COMPOSITE_SCORE'])}; font-size: 15px;">{c['COMPOSITE_SCORE']}%</td>
            </tr>
        """
    tables_html += "</table></div>"

html_code = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; padding: 20px; background-color: #f0f2f5; border-radius: 8px;">
    <div style="text-align: center; margin-bottom: 30px;">
        <h1 style="color: #1a237e; margin-bottom: 5px;">Data Quality Control Tower</h1>
        <p style="color: #5f6368; margin-top: 0; font-size: 16px;">Medallion Architecture: <b>Silver Layer</b> | Cleansed & Standardized Cleans</p>
    </div>
    <div style="background: white; padding: 25px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); margin-bottom: 30px; text-align: center; border: 1px solid #e0e0e0; border-top: 6px solid #1a73e8;">
        <h3 style="margin: 0; color: #5f6368; text-transform: uppercase; letter-spacing: 1px;">Overall Silver Composite Score</h3>
        <h1 style="margin: 15px 0 0 0; color: {get_color(silver_layer_score)}; font-size: 54px;">
            {round(silver_layer_score, 2)}%
        </h1>
        <p style="color: #5f6368; font-size: 14px; margin-top: 10px; font-weight: bold;">📊 Standardized Silver Tier Validation Complete.</p>
    </div>
    {tables_html}
</div>
"""

displayHTML(html_code)

Enter Snowflake Password:  [REDACTED]

Executing Silver Scans...
✅ Silver data successfully appended to Snowflake!


Column Name,Nulls,Unique,Valid,Anomalies,Composite Score
DESYNPUF_ID,✅ 0,"116,078","342,912",✅ 0,83.46%
BENE_BIRTH_DT,✅ 0,900,"342,912",✅ 0,75.07%
BENE_DEATH_DT,"337,638",36,"342,912",✅ 0,50.39%
BENE_SEX_IDENT_CD,✅ 0,2,"342,912",✅ 0,75.0%
BENE_RACE_CD,✅ 0,4,"342,912",✅ 0,75.0%
BENE_ESRD_IND,✅ 0,2,"342,912",✅ 0,75.0%
SP_STATE_CODE,✅ 0,52,"342,912",✅ 0,75.0%
BENE_COUNTY_CD,✅ 0,310,"342,912",✅ 0,75.02%
BENE_HI_CVRAGE_TOT_MONS,✅ 0,13,"342,912",✅ 0,75.0%
BENE_SMI_CVRAGE_TOT_MONS,✅ 0,13,"342,912",✅ 0,75.0%


In [0]:
from pyspark.sql.functions import col, sum as _sum, when, lit, countDistinct, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
import pandas as pd
import getpass

# ====================================================================
# 1. SNOWFLAKE CONNECTION & CONFIGURATION
# ====================================================================
sf_password = getpass.getpass("Enter Snowflake Password: ")
sfOptions = {
    "host": "iovnsqz-sf95102.snowflakecomputing.com",
    "sfUser": "madaladevendra",
    "sfPassword": sf_password,
    "sfDatabase": "HEALTHCARE_RCM",
    "sfSchema": "GOLD",
    "sfWarehouse": "RCM_WH"
}

LAYER_NAME = "Gold"

# ====================================================================
# 2. ENTERPRISE DATA QUALITY RULES (GOLD STAR SCHEMA VALIDATION)
# ====================================================================

def analyze_gold_composite_full(df, table_name, rules_config):
    total_rows = df.count()
    if total_rows == 0:
        return {"table": table_name, "score": 0.0, "total_rows": 0, "columns": []}

    # Identify columns explicitly mapped in rules (Critical)
    critical_actual_cols = {conf["actual_col"] for conf in rules_config}
    
    # Identify remaining columns as Non-Critical
    all_columns = df.columns
    non_critical_cols = [c for c in all_columns if c not in critical_actual_cols]

    agg_exprs = []
    
    # 1. Build expressions for Critical columns (with rules)
    for conf in rules_config:
        c_name = conf["column"]
        act_col = conf["actual_col"]
        rule_expr = conf["rule"]
        
        agg_exprs.append(_sum(when(rule_expr, 1).otherwise(0)).alias(f"{act_col}_anomalies"))
        agg_exprs.append(_sum(when(col(act_col).isNull(), 1).otherwise(0)).alias(f"{act_col}_nulls"))
        agg_exprs.append(countDistinct(col(act_col)).alias(f"{act_col}_uniques"))
        
    # 2. Build expressions for Non-Critical columns (Standard profiling, 0 anomalies)
    for c_name in non_critical_cols:
        agg_exprs.append(lit(0).alias(f"{c_name}_anomalies"))
        agg_exprs.append(_sum(when(col(c_name).isNull(), 1).otherwise(0)).alias(f"{c_name}_nulls"))
        agg_exprs.append(countDistinct(col(c_name)).alias(f"{c_name}_uniques"))
        
    metrics_row = df.select(*agg_exprs).collect()[0].asDict()
    
    col_scores = []
    
    # Process Critical Columns
    for conf in rules_config:
        c_name = conf["column"]
        act_col = conf["actual_col"]
        
        anomalies = metrics_row[f"{act_col}_anomalies"]
        nulls = metrics_row[f"{act_col}_nulls"]
        uniques = metrics_row[f"{act_col}_uniques"]
        valid_count = total_rows - anomalies 
        
        score_completeness = ((total_rows - nulls) / total_rows) * 100 if total_rows > 0 else 0
        score_uniqueness = (uniques / total_rows) * 100 if total_rows > 0 else 0
        score_yield = (valid_count / total_rows) * 100 if total_rows > 0 else 0
        score_accuracy = ((total_rows - anomalies) / total_rows) * 100 if total_rows > 0 else 0
        composite_score = (score_completeness + score_uniqueness + score_yield + score_accuracy) / 4
        
        col_scores.append({
            "column": c_name,
            "actual_col": act_col,
            "is_critical": "YES",
            "score": round(composite_score, 2),
            "anomalies": anomalies,
            "nulls": nulls,
            "uniques": uniques,
            "valid_count": valid_count,
            "completeness": round(score_completeness, 2),
            "uniqueness": round(score_uniqueness, 2),
            "yield": round(score_yield, 2),
            "accuracy": round(score_accuracy, 2),
            "breakdown": f"Comp:{round(score_completeness,1)}% | Uniq:{round(score_uniqueness,1)}% | Yld:{round(score_yield,1)}% | Acc:{round(score_accuracy,1)}%"
        })

    # Process Non-Critical Columns
    for c_name in non_critical_cols:
        anomalies = metrics_row[f"{c_name}_anomalies"]
        nulls = metrics_row[f"{c_name}_nulls"]
        uniques = metrics_row[f"{c_name}_uniques"]
        valid_count = total_rows - anomalies 
        
        score_completeness = ((total_rows - nulls) / total_rows) * 100 if total_rows > 0 else 0
        score_uniqueness = (uniques / total_rows) * 100 if total_rows > 0 else 0
        score_yield = (valid_count / total_rows) * 100 if total_rows > 0 else 0
        score_accuracy = ((total_rows - anomalies) / total_rows) * 100 if total_rows > 0 else 0
        composite_score = (score_completeness + score_uniqueness + score_yield + score_accuracy) / 4
        
        col_scores.append({
            "column": c_name,
            "actual_col": c_name,
            "is_critical": "NO",
            "score": round(composite_score, 2),
            "anomalies": anomalies,
            "nulls": nulls,
            "uniques": uniques,
            "valid_count": valid_count,
            "completeness": round(score_completeness, 2),
            "uniqueness": round(score_uniqueness, 2),
            "yield": round(score_yield, 2),
            "accuracy": round(score_accuracy, 2),
            "breakdown": f"Comp:{round(score_completeness,1)}% | Uniq:{round(score_uniqueness,1)}% | Yld:{round(score_yield,1)}% | Acc:{round(score_accuracy,1)}%"
        })
        
    table_score = sum([c["score"] for c in col_scores]) / len(col_scores) if col_scores else 100.0
    
    return {
        "table": table_name, 
        "score": round(table_score, 2), 
        "total_rows": total_rows, 
        "columns": col_scores
    }

print("Validating Star Schema Integrity on GOLD tables. Please wait...")

# --- 1. DIMENSION: BENEFICIARY ---
dim_bene_df = spark.table("healthcare_rcm.gold.dim_beneficiary")
dim_bene_rules = [
    {"column": "PK_beneficiary_key", "actual_col": "beneficiary_key", "rule": col("beneficiary_key").isNull()},
    {"column": "desynpuf_id", "actual_col": "desynpuf_id", "rule": col("desynpuf_id").isNull()}
]
dim_bene_dq = analyze_gold_composite_full(dim_bene_df, "DIM_BENEFICIARY", dim_bene_rules)

# --- 2. DIMENSION: PROVIDER ---
dim_prov_df = spark.table("healthcare_rcm.gold.dim_provider")
dim_prov_rules = [
    {"column": "PK_provider_key", "actual_col": "provider_key", "rule": col("provider_key").isNull()},
    {"column": "provider_id", "actual_col": "provider_id", "rule": col("provider_id").isNull()}
]
dim_prov_dq = analyze_gold_composite_full(dim_prov_df, "DIM_PROVIDER", dim_prov_rules)

# --- 3. FACT: INPATIENT ---
fact_inp_df = spark.table("healthcare_rcm.gold.fact_inpatient")
fact_inp_rules = [
    {"column": "PK_clm_id", "actual_col": "clm_id", "rule": col("clm_id").isNull()},
    {"column": "FK_beneficiary_key", "actual_col": "beneficiary_key", "rule": col("beneficiary_key").isNull()},
    {"column": "FK_provider_key", "actual_col": "provider_key", "rule": col("provider_key").isNull()},
    {"column": "claim_amount", "actual_col": "claim_amount", "rule": col("claim_amount") < 0}
]
fact_inp_dq = analyze_gold_composite_full(fact_inp_df, "FACT_INPATIENT", fact_inp_rules)

# --- 4. FACT: PHARMACY ---
fact_pde_df = spark.table("healthcare_rcm.gold.fact_pharmacy")
fact_pde_rules = [
    {"column": "PK_pde_id", "actual_col": "pde_id", "rule": col("pde_id").isNull()},
    {"column": "FK_beneficiary_key", "actual_col": "beneficiary_key", "rule": col("beneficiary_key").isNull()},
    {"column": "total_drug_cost", "actual_col": "total_drug_cost", "rule": col("total_drug_cost") < 0}
]
fact_pde_dq = analyze_gold_composite_full(fact_pde_df, "FACT_PHARMACY", fact_pde_rules)


# ====================================================================
# 3. WRITE TO SNOWFLAKE AUDIT TABLE WITH STRICT SCHEMA
# ====================================================================
flat_rows = []
all_tables_dq = [dim_bene_dq, dim_prov_dq, fact_inp_dq, fact_pde_dq]

for t in all_tables_dq:
    for c in t['columns']:
        flat_rows.append({
            "LAYER": LAYER_NAME,
            "DATASET_NAME": t['table'],
            "COLUMN_NAME": c['column'],
            "IS_CRITICAL": c['is_critical'],
            "TOTAL_ROWS": t['total_rows'],
            "NULL_COUNT": c['nulls'],
            "UNIQUE_COUNT": c['uniques'],
            "VALID_COUNT": c['valid_count'],
            "ANOMALY_COUNT": c['anomalies'],
            "COMPLETENESS_SCORE": c['completeness'],
            "UNIQUENESS_SCORE": c['uniqueness'],
            "YIELD_SCORE": c['yield'],
            "ACCURACY_SCORE": c['accuracy'],
            "COMPOSITE_SCORE": c['score']
        })

dq_schema = StructType([
    StructField("LAYER", StringType(), True),
    StructField("DATASET_NAME", StringType(), True),
    StructField("COLUMN_NAME", StringType(), True),
    StructField("IS_CRITICAL", StringType(), True),
    StructField("TOTAL_ROWS", LongType(), True),
    StructField("NULL_COUNT", LongType(), True),
    StructField("UNIQUE_COUNT", LongType(), True),
    StructField("VALID_COUNT", LongType(), True),
    StructField("ANOMALY_COUNT", LongType(), True),
    StructField("COMPLETENESS_SCORE", DoubleType(), True),
    StructField("UNIQUENESS_SCORE", DoubleType(), True),
    StructField("YIELD_SCORE", DoubleType(), True),
    StructField("ACCURACY_SCORE", DoubleType(), True),
    StructField("COMPOSITE_SCORE", DoubleType(), True)
])

dq_df = spark.createDataFrame(pd.DataFrame(flat_rows), schema=dq_schema).withColumn("AUDIT_TIMESTAMP", current_timestamp())

# Append to Snowflake
dq_df.write.format("snowflake").options(**sfOptions).option("dbtable", "ENTERPRISE_DQ_AUDIT_LOG").mode("append").save()
print("✅ Complete Gold Star Schema audit data (Critical + Non-Critical) successfully appended to Snowflake!")

# ====================================================================
# 4. CALCULATE LAYER LEVEL HEALTH SCORE
# ====================================================================
gold_layer_score = sum([t["score"] for t in all_tables_dq]) / len(all_tables_dq)

# ====================================================================
# 5. RENDER NATIVE DATABRICKS UI
# ====================================================================
def get_color(score):
    if score >= 85: return '#0D5C2F'  # Dark Green
    if score >= 77.77: return '#1e8e3e' # Standard Green
    return '#d93025'                  # Red

tables_html = ""
for t in all_tables_dq:
    tables_html += f"""
    <div style="background: white; padding: 15px; border-radius: 8px; border-top: 4px solid #fbbc04; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin-bottom: 25px;">
        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #eee; padding-bottom: 10px; margin-bottom: 10px;">
            <h3 style="margin: 0; color: #1a237e;">{t['table']} Dataset</h3>
            <h2 style="margin: 0; color: {get_color(t['score'])};">{t['score']}% Avg</h2>
        </div>
        <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 13px;">
            <tr style="background-color: #fff8e1;">
                <th style="padding: 8px; border-bottom: 1px solid #ddd; color: #b07d00;">Column Name</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: center;">Is Critical?</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Nulls</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Unique</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Valid</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Integrity Violations</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right; background: #e8f0fe;">4-Dim Percentage Breakdown</th>
                <th style="padding: 8px; border-bottom: 1px solid #ddd; text-align: right;">Composite Score (/4)</th>
            </tr>
    """
    for c in t['columns']:
        col_name_display = c['column']
        if col_name_display.startswith("PK_"): col_name_display = f"🔑 {col_name_display}"
        if col_name_display.startswith("FK_"): col_name_display = f"🔗 {col_name_display}"

        critical_badge = "<span style='color: #1e8e3e; font-weight: bold;'>YES</span>" if c['is_critical'] == "YES" else "<span style='color: #5f6368;'>NO</span>"
        anomaly_text = f"<span style='color: #1e8e3e;'>✅ 0</span>" if c['anomalies'] == 0 else f"<span style='color: #d93025;'>{c['anomalies']:,}</span>"
        null_text = f"<span style='color: #1e8e3e;'>✅ 0</span>" if c['nulls'] == 0 else f"<span style='color: #5f6368;'>{c['nulls']:,}</span>"

        tables_html += f"""
            <tr>
                <td style="padding: 8px; border-bottom: 1px solid #eee; font-family: monospace;"><strong>{col_name_display}</strong></td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: center;">{critical_badge}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right;">{null_text}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right; color: #1a73e8;">{c['uniques']:,}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right; color: #1e8e3e;">{c['valid_count']:,}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right;">{anomaly_text}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right; color: #5f6368; font-size: 11px; background: #f8f9fa;">{c['breakdown']}</td>
                <td style="padding: 8px; border-bottom: 1px solid #eee; text-align: right; font-weight: bold; color: {get_color(c['score'])}; font-size: 16px;">{c['score']}%</td>
            </tr>
        """
    tables_html += "</table></div>"

html_code = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; padding: 20px; background-color: #f0f2f5; border-radius: 8px;">
    
    <div style="text-align: center; margin-bottom: 30px;">
        <h1 style="color: #b07d00; margin-bottom: 5px;">Star Schema Data Quality Control Tower</h1>
        <p style="color: #5f6368; margin-top: 0; font-size: 16px;">Medallion Architecture: <b>Gold Layer</b> | Critical & Non-Critical Attribute Profiling</p>
    </div>
    
    <div style="background: white; padding: 25px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); margin-bottom: 30px; text-align: center; border: 1px solid #e0e0e0; border-top: 6px solid #fbbc04;">
        <h3 style="margin: 0; color: #5f6368; text-transform: uppercase; letter-spacing: 1px;">Overall Gold Composite Score</h3>
        <h1 style="margin: 15px 0 0 0; color: {get_color(gold_layer_score)}; font-size: 54px;">
            {round(gold_layer_score, 2)}%
        </h1>
        <p style="color: #1e8e3e; font-size: 14px; margin-top: 10px; font-weight: bold;">✅ Star Schema Complete: Critical Keys & Non-Critical Attributes Fully Audited.</p>
    </div>

    {tables_html}
    
</div>
"""

displayHTML(html_code)

Enter Snowflake Password:  [REDACTED]

Validating Star Schema Integrity on GOLD tables. Please wait...
✅ Complete Gold Star Schema audit data (Critical + Non-Critical) successfully appended to Snowflake!


Column Name,Is Critical?,Nulls,Unique,Valid,Integrity Violations,4-Dim Percentage Breakdown,Composite Score (/4)
🔑 PK_beneficiary_key,YES,✅ 0,"342,912","342,912",✅ 0,Comp:100.0% | Uniq:100.0% | Yld:100.0% | Acc:100.0%,100.0%
desynpuf_id,YES,✅ 0,"116,078","342,912",✅ 0,Comp:100.0% | Uniq:33.9% | Yld:100.0% | Acc:100.0%,83.46%
gold_processing_timestamp,NO,✅ 0,1,"342,912",✅ 0,Comp:100.0% | Uniq:0.0% | Yld:100.0% | Acc:100.0%,75.0%
Column Name,Is Critical?,Nulls,Unique,Valid,Integrity Violations,4-Dim Percentage Breakdown,Composite Score (/4)
🔑 PK_provider_key,YES,✅ 0,"6,818","6,818",✅ 0,Comp:100.0% | Uniq:100.0% | Yld:100.0% | Acc:100.0%,100.0%
provider_id,YES,✅ 0,"6,818","6,818",✅ 0,Comp:100.0% | Uniq:100.0% | Yld:100.0% | Acc:100.0%,100.0%
provider_type,NO,✅ 0,1,"6,818",✅ 0,Comp:100.0% | Uniq:0.0% | Yld:100.0% | Acc:100.0%,75.0%
gold_processing_timestamp,NO,✅ 0,1,"6,818",✅ 0,Comp:100.0% | Uniq:0.0% | Yld:100.0% | Acc:100.0%,75.0%
Column Name,Is Critical?,Nulls,Unique,Valid,Integrity Violations,4-Dim Percentage Breakdown,Composite Score (/4)
🔑 PK_clm_id,YES,✅ 0,"66,293","66,293",✅ 0,Comp:100.0% | Uniq:100.0% | Yld:100.0% | Acc:100.0%,100.0%


In [0]:
# Note: This cell is not needed - each layer (Bronze/Silver/Gold) already writes
# its own audit logs to Snowflake within their respective cells above.
# The variables 'combined_gold_df' and 'snowflake_options' are not defined in this notebook.